In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from daemon_analysis_tools.io.csv_handler import load_and_process_csv
from daemon_analysis_tools.io.yaml_handler import save_answers_to_yaml, load_answers_from_yaml
from daemon_analysis_tools.processing.grouper import group_questions_by_journal
from daemon_analysis_tools.services.discrepancy_resolver import resolve_discrepancy

Load and process data:
- Group answers by publisher and journal, trying to uniform names written in slightly different ways.
- Store in a DataFrame

In [14]:
data = load_and_process_csv("../../data/raw/rdp.csv")

Get a `dict` labeled by publisher names of `dict`s labeled by journal names of `dict`s of `Question` instances. The `.answer` attribute contains the answers given by the respondents and the explanations text to motivate it.

In [15]:
question_metadata_file = "../../data/metadata/question_metadata.yaml"

grouped_questions = group_questions_by_journal(data, question_metadata_file)

## Resolve discrepancies

The `Question` class has a `.resolve_discrepancies` method which updates `Question.anwsers` with the correct answer.

For example, let's consider IOP's 2D Materials. Question 7 has discrepancies.

In [16]:
for journal, data in grouped_questions["codata"].items():
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies():
            answer.print_qa()
    print("\n\n")

data_science_journal
7. Timing of data release
  Resp. 0:
    Answer: Required data must be available after official publication.
    Explanation: The journal strongly encourages authors to make all data associated with their submission openly available, according to the FAIR principles (Findable, Accessible, Interoperable, Reusable). This should be linked to from a Data Accessibility Statement within the submitted paper, which will be made public upon publication. If data is not being made available with the journal publication then ideally a statement from the author should be provided within the submission to explain why. (https://datascience.codata.org/about/reproducibility)
  Resp. 1:
    Answer: Required data must be available prior to official publication.
    Explanation: The journal strongly encourages authors to make all data associated with their submission openly available, according to the FAIR principles (Findable, Accessible, Interoperable, Reusable). This should be link

Inconsistencies can be removed manually, passing the index of the correct respondent.

In [17]:
for j in ["data_science_journal"]:
    for i in [9, 10]:
        resolve_discrepancy(
            grouped_questions["codata"][j][i],
            correct_answer=0,
            discrepancy_reason="Text not found",
        )
    
    for i in [7]:
        resolve_discrepancy(
            grouped_questions["codata"][j][i],
            correct_answer=0,
            discrepancy_reason="Language understanding",
        )

    for i in [11, 12]:
        resolve_discrepancy(
            grouped_questions["codata"][j][i],
            correct_answer=0,
            discrepancy_reason="Formatting",
        )


In [18]:
for journal, data in grouped_questions["codata"].items():
    print("#############################################################")
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies() and answer.correct_answer is None:
            answer.print_qa()

#############################################################
data_science_journal


In [19]:
save_answers_to_yaml(
    grouped_questions,
    parent_folder="../../data/processed/all_answers",
    save_only=["codata"],
)

After doing this, the `.get_final_answer()` method returns the correct answer.